# Comparing quantum values

Preserve both original examples: equality of signed/unsigned words initialized to zero, and a coherent unsigned threshold. Sampling is local and seeded; no cloud result is claimed.


In [ ]:
from collections import Counter
from random import Random
from typing import Annotated

import flagquantum as fq
from flagquantum.synth import Output, QNum, allocate, hadamard_transform, qfunc, synthesize


## Equal words

A five-bit signed zero and three-bit unsigned zero compare equal.


In [ ]:
@qfunc
def equal_words(a: Output[Annotated[QNum, (5, True, 0)]], b: Output[Annotated[QNum, 3]], res: Output[QNum]) -> None:
    allocate(a)
    allocate(b)
    res |= a == b


In [ ]:
equal_words_program = synthesize(equal_words)
equal_words_result = fq.run(equal_words_program)
equal_words_probabilities = dict(equal_words_program.numeric_probabilities(equal_words_result, registers=('a', 'b', 'res')))
equal_words_counts = Counter(Random(17).choices(list(equal_words_probabilities), weights=list(equal_words_probabilities.values()), k=1000))
assert sum(equal_words_counts.values()) == 1000
print("Probabilities:", equal_words_probabilities)
print("Counts:", dict(equal_words_counts))
assert equal_words_probabilities == {(0, 0, 1): 1.0}


## Coherent threshold

All eight values remain coherent. Values 0, 1 and 2 carry result one; the others carry zero. No measurement occurs during the comparison.


In [ ]:
@qfunc
def threshold(a: Output[QNum], res: Output[QNum]) -> None:
    allocate(3, a)
    hadamard_transform(a)
    res |= a <= 2


In [ ]:
threshold_program = synthesize(threshold)
threshold_result = fq.run(threshold_program)
threshold_probabilities = dict(threshold_program.probabilities(threshold_result, registers=('a', 'res')))
threshold_counts = Counter(Random(17).choices(list(threshold_probabilities), weights=list(threshold_probabilities.values()), k=1000))
assert sum(threshold_counts.values()) == 1000
print("Probabilities:", threshold_probabilities)
print("Counts:", dict(threshold_counts))
expected = {(a, int(a <= 2)): 1/8 for a in range(8)}
assert threshold_probabilities.keys() == expected.keys()
assert all(abs(threshold_probabilities[key]-value) < 1e-12 for key,value in expected.items())
